# 🛩️ Example: Working with ImprovedLAPANEnv

This notebook demonstrates the usage of the improved LAPAN LSU-05 NG environment with normalized spaces.

## 📋 Features of ImprovedLAPANEnv:
- ✅ Normalized action and observation spaces [-1, 1]
- ✅ Improved reward function
- ✅ Optimized for reinforcement learning

## 📚 Importing Libraries

In [1]:
import gymnasium as gym
import numpy as np
import tensoraerospace
from tensoraerospace.utils import generate_time_period, convert_tp_to_sec_tp
from tensoraerospace.signals.standard import unit_step

## ⚙️ Parameters Setup

In [ ]:
dt = 0.01
tp = generate_time_period(tn=20, dt=dt)
number_time_steps = len(tp)
reference_signals = np.reshape(
    unit_step(degree=5, tp=tp, time_step=10*dt, output_rad=True), 
    [1, -1]
)
print(f"📊 Parameters: time={tp[-1]:.1f}s, steps={number_time_steps}")

## 🚀 Creating the Environment

In [3]:
# Initial state [u, w, q, theta]
initial_state = np.array([0.0, 0.0, 0.0, 0.0])

env = gym.make(
    "ImprovedLAPAN-v0",
    initial_state=initial_state,
    reference_signal=reference_signals,
    number_time_steps=number_time_steps,
    dt=dt,
    initial_elevator_deg=0.0
)

obs, info = env.reset()
print(f"✅ Environment created!")
print(f"📊 Observation: {obs}")
print(f"🎯 Actions: {env.action_space}")
print(f"📐 Observations: {env.observation_space}")

✅ Environment created!
📊 Observation: [0. 0. 0. 0.]
🎯 Actions: Box(-1.0, 1.0, (1,), float32)
📐 Observations: Box(-1.0, 1.0, (4,), float32)


## 🎮 Executing a Step

In [4]:
action = np.array([0.5], dtype=np.float32)
obs, reward, terminated, truncated, info = env.step(action)
print(f"🎯 Action: {action[0]:.2f}")
print(f"📊 Observation: {obs}")
print(f"🏆 Reward: {reward:.6f}")

🎯 Action: 0.50
📊 Observation: [0. 0. 0. 0.]
🏆 Reward: -0.000000


## Visualization: Full Episode with Proportional Control

Running a complete episode and plotting the LAPAN LSU-05 response:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create a fresh environment for the full episode
initial_state_vis = np.array([0.0, 0.0, 0.0, 0.0])
tp_vis = generate_time_period(tn=20, dt=0.01)
number_time_steps_vis = len(tp_vis)
reference_signals_vis = np.reshape(
    unit_step(degree=5, tp=tp_vis, time_step=10*dt, output_rad=True),
    [1, -1]
)

env_vis = gym.make(
    "ImprovedLAPAN-v0",
    initial_state=initial_state_vis,
    reference_signal=reference_signals_vis,
    number_time_steps=number_time_steps_vis,
    dt=0.01,
    initial_elevator_deg=0.0
)

obs, _ = env_vis.reset()
observations, rewards, actions = [], [], []

done = False
while not done:
    # Simple proportional control: drive pitch error (obs[0]) toward zero
    action = np.array([np.clip(-2.0 * obs[0], -1, 1)], dtype=np.float32)
    obs, reward, terminated, truncated, _ = env_vis.step(action)
    done = terminated or truncated
    observations.append(obs.copy())
    rewards.append(reward)
    actions.append(action[0])

observations = np.array(observations)
time = np.arange(len(rewards)) * 0.01

# Create professional multi-panel figure
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('LAPAN LSU-05 NG -- Proportional Control Response', fontsize=14, fontweight='bold')

# Plot 1: Pitch tracking error
axes[0, 0].plot(time, observations[:, 0], 'b-', linewidth=1.5, label='Pitch error (norm)')
axes[0, 0].axhline(y=0, color='r', linestyle='--', alpha=0.5, label='Target')
axes[0, 0].fill_between(time, -0.05, 0.05, alpha=0.1, color='green', label='$\\pm$5% band')
axes[0, 0].set_xlabel('Time (s)')
axes[0, 0].set_ylabel('Normalized error')
axes[0, 0].set_title('Pitch Tracking Error')
axes[0, 0].legend(fontsize=8)
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: State variables
axes[0, 1].plot(time, observations[:, 1], 'g-', linewidth=1.5, label='Pitch rate (norm)')
axes[0, 1].plot(time, observations[:, 2], 'm-', linewidth=1.5, label='Pitch angle (norm)')
axes[0, 1].set_xlabel('Time (s)')
axes[0, 1].set_ylabel('Normalized value')
axes[0, 1].set_title('State Variables')
axes[0, 1].legend(fontsize=8)
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Control action (elevator)
axes[1, 0].plot(time, actions, 'r-', linewidth=1.5)
axes[1, 0].set_xlabel('Time (s)')
axes[1, 0].set_ylabel('Action (normalized)')
axes[1, 0].set_title('Control Input (Elevator)')
axes[1, 0].set_ylim(-1.1, 1.1)
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Cumulative reward
cumulative = np.cumsum(rewards)
axes[1, 1].plot(time, cumulative, 'k-', linewidth=1.5)
axes[1, 1].set_xlabel('Time (s)')
axes[1, 1].set_ylabel('Cumulative reward')
axes[1, 1].set_title(f'Cumulative Reward (total: {cumulative[-1]:.1f})')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

env_vis.close()